## Use SHAP to map mortality drivers

Fit a model to all data, then use SHAP to derive explanations.

In [ ]:
import xarray as xr
import rioxarray
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import lightgbm as lgb
import fasttreeshap
import warnings
import logging
from tqdm.autonotebook import tqdm

import const
from gbm import split_xy, make_zif_quantile_estimator, get_results, balance_zeros, safe_logit

In [ ]:
warnings.filterwarnings("ignore", message="X does not have valid feature names")
logging.getLogger().setLevel(logging.CRITICAL)

In [ ]:
westmort = xr.open_zarr("../data_working/westmort.zarr/").compute().rio.write_crs(const.PROJECTION)
westmort

In [ ]:
def make_data_frame(agent: str, use_def: bool=True, use_vpd: bool=True) -> pd.DataFrame:
    agent_ba = f"{agent}_ba"
    agent_mort = f"{agent}_mort"
    agent_target = f"{agent}_target"
    
    cols_to_select = const.GBM_COVARIATES["hydro"] +\
        const.GBM_COVARIATES["topo"] +\
        const.GBM_COVARIATES["climate"] +\
        const.GBM_COVARIATES["structure"] +\
        [agent_ba, agent_mort, agent_target]

    if not use_def:
        cols_to_select.remove("def")
    if not use_vpd:
        cols_to_select.remove("vpd")

    westmort_df = westmort[cols_to_select].to_dataframe()
    westmort_df[agent_mort] = safe_logit(westmort_df[agent_mort])
    westmort_df = westmort_df[westmort_df[agent_ba] > 0].dropna()

    return westmort_df

In [ ]:
results = []
agents = list(const.HOST_DCA_CODES.keys())
gbm_args = {
    "learning_rate": 0.1,
    "n_estimators": 100,
    "num_leaves": 16,
    "bagging_fraction": 0.8,
    "feature_fraction": 0.8
}
sample_fraction = 0.1
vpd_def_args = [
    ("both", dict(use_def=True, use_vpd=True)),
    ("vpd_only", dict(use_def=False, use_vpd=True)),
    ("def_only", dict(use_def=True, use_vpd=True))
]

for name, args in tqdm(vpd_def_args):
    for agent in tqdm(agents):
        target_var = f"{agent}_target"
        
        df = make_data_frame(agent, **args)
        # print(train.columns)
        model = make_zif_quantile_estimator(gbm_args, gbm_args) # same args for classifier and regressor
    
        df_subsample = balance_zeros(df, target_var)
        
        X_subsample, y_subsample = split_xy(df_subsample, target_var)
        X, y = split_xy(df, target_var)
    
        model.fit(X_subsample, y_subsample)
        y_hat = model.predict(X)
        
        model_perf = get_results(y, y_hat)
    
        # Here we are specifically looking at the logits of the regressor arm.
        # The property regressor_ gives us the fitted object, while regressor
        # (without the underscore) gives us the unfitted object.
        # Also, only consider 10% of available data for SHAP for computational efficiency.
        X_sample = X.sample(frac=sample_fraction, random_state=8052026)
        explainer = fasttreeshap.TreeExplainer(model.regressor_.regressor_, algorithm="auto")
        shap_values = pd.DataFrame(
            data=explainer(X_sample).values,
            columns=X_sample.columns,
            index=X_sample.index
        )
    
        results.append({
            "expt": name,
            "agent": agent,
            "model": model,
            "X": X_subsample,
            "performance": model_perf,
            "shap": shap_values
        })

In [ ]:
both = pd.concat(
    [
        pd.merge(
            r["X"][["vpd", "def"]],
            r["shap"][["vpd", "def"]],
            suffixes=("_data", "_shap"),
            left_index=True,
            right_index=True
        )
        for r in results if r["expt"] == "both"
    ],
    axis=0,
    ignore_index=True
)

vpd_only = pd.concat(
    [
        pd.merge(
            r["X"][["vpd"]],
            r["shap"][["vpd"]],
            suffixes=("_data", "_shap"),
            left_index=True,
            right_index=True
        )
        for r in results if r["expt"] == "vpd_only"
    ],
    axis=0,
    ignore_index=True
)

def_only = pd.concat(
    [
        pd.merge(
            r["X"][["def"]],
            r["shap"][["def"]],
            suffixes=("_data", "_shap"),
            left_index=True,
            right_index=True
        )
        for r in results if r["expt"] == "def_only"
    ],
    axis=0,
    ignore_index=True
)

In [ ]:
import mpl_scatter_density
import matplotlib
import scienceplots

plt.style.use(["science", "no-latex", "nature"])

PAPER_RC = {
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "ytick.labelsize": 12,
    "xtick.labelsize": 12,
    "legend.fontsize": 12    
}

plt.rcParams.update(PAPER_RC)

In [ ]:
fig, axes = plt.subplots(
    2, 3, 
    subplot_kw=dict(projection="scatter_density"), 
    sharex="row",
    figsize=(8, 5),
    width_ratios=(1, 1, 0.3)
)

d = axes[0, 0].scatter_density(
    both["vpd_data"], both["vpd_shap"], 
    norm=matplotlib.colors.LogNorm(vmin=1, vmax=1000, clip=False)
)
axes[0, 0].set_title("VPD (kPa; both in model)")

axes[1, 0].scatter_density(
    both["def_data"], both["def_shap"], 
    norm=matplotlib.colors.LogNorm(vmin=1, vmax=1000, clip=False)
)
axes[1, 0].set_title("DEF (mm; both in model)")

axes[0, 1].scatter_density(
    vpd_only["vpd_data"], vpd_only["vpd_shap"], 
    norm=matplotlib.colors.LogNorm(vmin=1, vmax=1000, clip=False)
)
axes[0, 1].set_title("VPD (kPa; DEF excluded from model)")

axes[1, 1].scatter_density(
    def_only["def_data"], def_only["def_shap"], 
    norm=matplotlib.colors.LogNorm(vmin=1, vmax=1000, clip=False)
)
axes[1, 1].set_title("DEF (mm; VPD excluded from model)")

for ax in axes.flat:
    ax.set_ylim(-1.0, 1.0)
    ax.grid()

for ax in axes[:, 2]:
    ax.axis("off")

fig.colorbar(d, ax=axes[:, 2], label="Points per pixel")
fig.supylabel("SHAP value (logits)")
plt.tight_layout()
plt.show()

## Build xarray object of SHAP values

SHAP gives us a data frame only at indices where we have nonzero host BA and no otherwise missing values. To use xarray for mapping, we have to convert this back to an xarray dataset. This doesn't work with a naive `to_xarray()` because not all values of x/y coordinates in the original data were present for modeling. First, we build a "template" DF with all the indices from the full xarray dataset, join onto that, and *then* convert to xarray. This eats a ton of RAM.

In [ ]:
template_df = westmort["def"].to_dataframe().drop(columns=["spatial_ref", "def"])

In [ ]:
shap_xarray_objs = [
    template_df.join(results[agent]["shap"], how="left").to_xarray()\
        .rename({f"{agent}_ba":"agent_ba", f"{agent}_mort":"agent_mort"})\
        .to_dataarray()\
        .assign_coords(agent=agent)
    for agent in results
]

In [ ]:
shap_xarray = xr.concat(shap_xarray_objs, "agent")

In [ ]:
shap_xarray.to_zarr("../data_working/gbm_shap_new.zarr")

In [ ]:
del shap_xarray
# del shap_xarray_objs

In [ ]:
pd.json_normalize([results[key]["performance"] for key in results])